# 15 - Regulatory Researcher
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent-use-cases/blob/main/examples/15-regulatory-researcher/regulatory_researcher_workbook.ipynb)

Extract every obligation, deadline, and penalty from a regulation -- with a hard rule that every finding must cite its exact source article.

**Harness focus:** Citation-grounded extraction -- no finding without a source article

In [ ]:
try:
    from google.colab import userdata  # noqa: F401
    import subprocess
    subprocess.run(["pip", "install", "-q", "langchain-openai", "pydantic", "python-dotenv"], check=True)
    print("Packages installed.")
except ImportError:
    print("Local environment.")

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    from dotenv import load_dotenv
    load_dotenv()
    if not os.getenv("OPENAI_API_KEY"):
        raise EnvironmentError("Set OPENAI_API_KEY in .env or Colab Secrets.")
print("API key ready.")

## Part 1: Why citation matters in regulatory extraction

Without a source citation, a compliance finding cannot be:
- Traced back to the regulation for verification
- Used to draft a compliance policy with authority
- Challenged if the model hallucinated it

The pattern: every obligation and penalty in the schema has a `source_article` field. The system prompt contains a hard rule: if you cannot cite an exact article, do not include the finding.

This makes the output **auditable** -- a compliance team can check every item against the regulation in seconds.

## Part 2: Schema -- source_article on every finding

In [ ]:
from typing import List, Optional
from pydantic import BaseModel, Field

class RegulatoryObligation(BaseModel):
    source_article: str = Field(
        description="Exact article reference, e.g. 'Article 4(1)'"
    )
    obligation: str
    applies_to: str
    is_ongoing: bool
    deadline: Optional[str] = None

class RegulatoryPenalty(BaseModel):
    source_article: str
    trigger: str
    maximum_fine: Optional[str] = None
    other_consequences: List[str]

class ComplianceSummary(BaseModel):
    regulation_name: str
    jurisdiction: str
    in_force_date: Optional[str] = None
    obligations: List[RegulatoryObligation]
    key_deadlines: List[str]
    penalties: List[RegulatoryPenalty]
    high_priority_gaps: List[str]

print("Schema defined.")

## Part 3: System prompt -- the citation rule

The citation rule is the key constraint in this harness. It tells the model: cite the exact article or omit the finding.

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

RESEARCHER_SYSTEM = SystemMessage("""
You are a regulatory compliance analyst.

Extract every obligation, deadline, and penalty from the regulation text.

CITATION RULE (mandatory):
  Every item in obligations and penalties MUST include the exact source_article
  (e.g. 'Article 5(1)(f)'). If you cannot point to a specific article or section
  in the provided text, do not include the finding.
  Cite precisely; never paraphrase article numbers.

key_deadlines: each must include the article reference, e.g.
  'Article 15(2): quarterly report within 30 days of quarter end'
""")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
researcher = RESEARCHER_SYSTEM | llm.with_structured_output(ComplianceSummary)
print("Agent ready.")

## Part 4: Run on the DMCR 2024 excerpt

In [ ]:
DMCR_TEXT = """
DIGITAL MARKETS CONDUCT REGULATION 2024 (DMCR 2024)
Jurisdiction: United Kingdom  |  In force: 1 January 2025

Article 4 -- Data Portability
4(1) A designated undertaking shall provide end-users with the ability to export
their personal data in a machine-readable format within 30 days of a written request.
4(2) The export mechanism must be continuously available.

Article 7 -- Interoperability
7(1) A designated undertaking shall ensure third-party interoperability on FRAND terms.
7(2) Technical specifications must be published within 90 days of designation.

Article 11 -- Self-Preferencing Prohibition
11(1) A designated undertaking shall not rank its own products more favourably than
third parties in search results or recommendations.

Article 15 -- Periodic Reporting
15(1) A quarterly compliance report shall be submitted to the Authority.
15(2) Each report must be submitted within 30 days of the end of the relevant quarter.

Article 22 -- Financial Penalties
22(1) Breach: up to 10% of total worldwide annual turnover.
22(2) Repeated infringement: up to 20% of total worldwide annual turnover.

Article 23 -- Periodic Penalty Payments
23(1) Up to 5% of average daily worldwide turnover per day of non-compliance.
"""
print("Regulation text loaded.")

In [ ]:
result = researcher.invoke(HumanMessage(content="Regulation text to analyse:\n\n" + DMCR_TEXT))
print(f"Regulation: {result.regulation_name}")
print(f"Obligations: {len(result.obligations)} | Penalties: {len(result.penalties)}")
print("\nObligations with citations:")
for o in result.obligations:
    print(f"  [{o.source_article}] {o.obligation[:70]}")

## Part 5: Compare outputs with vs without the citation rule

In [ ]:
NO_CITATION_SYSTEM = SystemMessage("""
You are a regulatory compliance analyst.
Extract every obligation, deadline, and penalty from the regulation text.
Fill source_article with your best guess if you are unsure.
""")

researcher_no_cite = NO_CITATION_SYSTEM | llm.with_structured_output(ComplianceSummary)
result_nc = researcher_no_cite.invoke(
    HumanMessage(content="Regulation text to analyse:\n\n" + DMCR_TEXT)
)

print("WITH citation rule:")
for o in result.obligations:
    print(f"  [{o.source_article}] {o.obligation[:60]}")

print("\nWITHOUT citation rule (model may hallucinate or omit refs):")
for o in result_nc.obligations:
    print(f"  [{o.source_article}] {o.obligation[:60]}")

## Exercise: Add a compliance_checklist field

Add `compliance_checklist: List[str]` to `ComplianceSummary`. Each item should be an actionable step derived from one obligation, including the article reference.

Example: `'Implement data export API within 30 days of request (Article 4(1))'`

**Starter:** Add the field below.

In [ ]:
# Starter: add compliance_checklist
class ComplianceSummaryV2(BaseModel):
    regulation_name: str
    jurisdiction: str
    in_force_date: Optional[str] = None
    obligations: List[RegulatoryObligation]
    key_deadlines: List[str]
    penalties: List[RegulatoryPenalty]
    high_priority_gaps: List[str]
    compliance_checklist: List[str]  # NEW

# TODO: update RESEARCHER_SYSTEM to instruct checklist generation
# TODO: rebuild researcher_v2 = updated_system | llm.with_structured_output(ComplianceSummaryV2)
# TODO: run on DMCR_TEXT and print compliance_checklist

In [ ]:
# Answer key
RESEARCHER_SYSTEM_V2 = SystemMessage("""
You are a regulatory compliance analyst.

Extract obligations, deadlines, and penalties with exact article citations.

CITATION RULE: every item must cite its exact source_article. No citation = no finding.

compliance_checklist: for each obligation, generate one actionable checklist item
in the format: '<action> (<article>)'.
""")

researcher_v2 = RESEARCHER_SYSTEM_V2 | llm.with_structured_output(ComplianceSummaryV2)
result_v2 = researcher_v2.invoke(
    HumanMessage(content="Regulation text to analyse:\n\n" + DMCR_TEXT)
)
print(f"Checklist ({len(result_v2.compliance_checklist)} items):")
for item in result_v2.compliance_checklist:
    print(f"  [ ] {item}")

## What You Built

- A citation-grounded extractor: every finding cites its source article
- `source_article` as a mandatory field on every extraction type
- A system prompt citation rule that prevents unsupported assertions
- A comparison showing the difference with and without the citation constraint

**Next steps:** Try example 16 (effort-impact ranking) or apply this pattern to a contract clause extractor or terms-of-service analyser.